In [1]:
import glob
import gzip
import re
import sys
from datetime import datetime

import joblib
import pandas as pd
from sklearn.ensemble import IsolationForest

In [3]:

MODEL_FILE = "ueba_isolation_forest.pkl"

# Auth.log shakliga mos Regex pattern
log_pattern = re.compile(
    r"^(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})(?:\.\d+)?(?:[+-]\d{2}:\d{2})\s+(\S+)\s+(\S+?)(?:\[\d+\])?: (.*)$"
)

data = []

# Tizimdagi barcha eski va yangi auth loglarni topamiz
log_files = glob.glob("/var/log/auth.log*")

for file_path in log_files:
    if file_path.endswith(".gz"):
        f = gzip.open(file_path, "rt", encoding="utf-8", errors="ignore")  # noqa: SIM115
    else:
        f = open(file_path, "r", encoding="utf-8", errors="ignore")  # noqa: SIM115

    with f:
        for line in f:
            match = log_pattern.match(line)
            if match:
                time_str, hostname, process, message = match.groups()

                # Faqat xavfsizlikka aloqador muhim voqealarni saralaymiz
                if any(
                    k in message
                    for k in ["session", "Accepted", "Failed", "polkitd", "sudo"]
                ):
                    user_match = re.search(r"for (?:user )?(\w+)", message)
                    if user_match:
                        user = user_match.group(1)
                    elif "by " in message:
                        user = message.split("by ")[-1].split()[0]
                    else:
                        user = "system"

                    dt = datetime.strptime(time_str, "%Y-%m-%dT%H:%M:%S")  # noqa: DTZ007
                    status = 0 if "Failed" in message else 1

                    data.append({
                        "datetime": dt,
                        "user": user,
                        "hour": dt.hour,
                        "day_of_week": dt.weekday(),
                        "status": status,
                    })

if not data:
    print("[!] Loglarda ma'lumot topilmadi.")
    sys.exit()

# DataFrame yaratamiz
df = pd.DataFrame(data)

In [19]:
# print('test1')
# df.sort_values(by="user", ascending=False).head(20)

In [21]:
# Feature Engineering: 1 soat ichidagi harakatlar sonini hisoblash
df["login_count_1h"] = df.groupby(
    ["user", pd.Grouper(key="datetime", freq="1h")]
)["status"].transform("count")
df


,datetime,user,hour,day_of_week,status,login_count_1h
1002,2026-07-16 09:34:03,action,9,3,1,1
3322,2026-08-01 17:12:47,action,17,5,1,1
1041,2026-07-16 10:25:11,action,10,3,1,5
2745,2026-07-28 09:07:42,action,9,1,1,1
1211,2026-07-17 10:51:01,action,10,4,1,2
...,...,...,...,...,...,...
979,2026-07-15 17:15:50,unix,17,2,1,1
42,2026-07-13 06:09:04,unix,6,0,1,2
450,2026-07-13 17:18:04,unix,17,0,1,5
464,2026-07-13 17:18:31,unix,17,0,1,5


In [26]:


# Model o'rganishi uchun belgilar (Features)
features = ["hour", "day_of_week", "status", "login_count_1h"]
X = df[features]

print("[INFO] Model yangi ma'lumotlar bo'yicha o'qitilmoqda...")

# Modelni har safar yangi loglar mezoniga ko'ra o'qitamiz
model = IsolationForest(contamination=0.05, random_state=42)
model.fit(X)

# Yangilangan bilimlarni keyingi ishlatish uchun saqlab qo'yamiz
joblib.dump(model, MODEL_FILE)
print(f"[INFO] Yangi xotira '{MODEL_FILE}' fayliga saqlandi.")

# Anomaliya va Risk Score hisoblash
df["anomaly_raw"] = model.predict(X)
df["score"] = model.decision_function(X)

min_score, max_score = df["score"].min(), df["score"].max()

if max_score != min_score:
    df["risk_score"] = (
        (max_score - df["score"]) / (max_score - min_score) * 100
    ).round(1)
else:
    df["risk_score"] = 0


[INFO] Model yangi ma'lumotlar bo'yicha o'qitilmoqda...
[INFO] Yangi xotira 'ueba_isolation_forest.pkl' fayliga saqlandi.


In [28]:

# Hisobotni ekranga chiqarish
print("\n=== UEBA ANOMALIYALARNI ANIQLASH HISOBOTI ===")
anomalies = df[df["anomaly_raw"] == -1].sort_values(
    by="risk_score", ascending=False
)
anomalies


=== UEBA ANOMALIYALARNI ANIQLASH HISOBOTI ===


,datetime,user,hour,day_of_week,status,login_count_1h,anomaly_raw,score,risk_score
1499,2026-07-19 00:12:10,unix,0,6,1,1,-1,-0.076612,100.0
1493,2026-07-19 00:12:09,ahmadjon,0,6,1,2,-1,-0.072309,98.7
1495,2026-07-19 00:12:09,ahmadjon,0,6,1,2,-1,-0.072309,98.7
1497,2026-07-19 00:12:09,system,0,6,1,4,-1,-0.061806,95.4
1498,2026-07-19 00:12:09,system,0,6,1,4,-1,-0.061806,95.4
1490,2026-07-19 00:12:01,sddm,0,6,1,4,-1,-0.061806,95.4
1496,2026-07-19 00:12:09,sddm,0,6,1,4,-1,-0.061806,95.4
1491,2026-07-19 00:12:01,system,0,6,1,4,-1,-0.061806,95.4
1500,2026-07-19 00:12:19,sddm,0,6,1,4,-1,-0.061806,95.4
1492,2026-07-19 00:12:01,sddm,0,6,1,4,-1,-0.061806,95.4
